# 40.21 Остаточный сигнал-кандидат ТТРКГ

Из измеренного ТТРКГ вычитается модельный вклад мягких тканей и лёгкого. Выход
называется только остатком-кандидатом: сердечное происхождение не установлено.
Ноутбук не использует внутрисессионную SE как полную погрешность и не работает
без отдельного принятого бюджета дисперсий и взаимной ковариации для каждого
точного ключа записи.


In [ ]:
import hashlib
import json
import os
from pathlib import Path

import numpy as np

from ttrkg_analysis import residual_candidate

test = residual_candidate([0.03, 0.02], [0.01, -0.01], [4e-6, 4e-6], [1e-6, 1e-6], [0.0, 0.0])
np.testing.assert_allclose(test["residual"], [0.02, 0.03])
REAL_MODE = os.environ.get("KALMYKOV_RUN_REAL", "0") == "1"
print("40.21 synthetic_self_test: passed")


In [ ]:
KEY_FIELDS = ['experiment_id', 'subject_id', 'record_id', 'configuration_id', 'montage_id', 'side_montage_id', 'side_size_mm', 'channel_state', 'mode']


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def key(item):
    return tuple(item.get(field) for field in KEY_FIELDS)


if not REAL_MODE:
    print("40.21 real_data_status: blocked_until_accepted_total_uncertainty_budget")
else:
    target = os.environ.get("KALMYKOV_TARGET_EXPERIMENT")
    if target not in {"exp02", "exp03"}:
        raise RuntimeError("KALMYKOV_TARGET_EXPERIMENT должен быть exp02 или exp03")
    config_path = Path(os.environ["KALMYKOV_EXP02_CONFIG" if target == "exp02" else "KALMYKOV_EXP03_CONFIG"]).expanduser().resolve()
    config = json.loads(config_path.read_text(encoding="utf-8"))
    spec = config.get("residual_analysis", {})
    if spec.get("uncertainty_status") != "accepted" or not spec.get("uncertainty_manifest"):
        raise RuntimeError("Не принят полный бюджет неопределённости остатка")
    analysis_dir = Path(config["derived_root"]).expanduser().resolve() / target / "analysis"
    measured_path = analysis_dir / ("40.01_ttrkg_ensembles.json" if target == "exp02" else "40.11_ttrkg_ensembles.json")
    predicted_path = analysis_dir / "40.20_tissue_prediction.json"
    measured = json.loads(measured_path.read_text(encoding="utf-8"))
    predicted = json.loads(predicted_path.read_text(encoding="utf-8"))
    if measured.get("status") != "accepted_input_conditional_ttrkg_ensembles" or not measured.get("ensembles"):
        raise RuntimeError("Неподходящий измеренный артефакт")
    allowed_prediction_status = {"same_experiment_conditional_known_tissue_prediction", "preliminary_cross_session_known_tissue_prediction"}
    if predicted.get("status") not in allowed_prediction_status or not predicted.get("predictions"):
        raise RuntimeError("Неподходящий модельный артефакт")
    measured_by_key = {key(item): item for item in measured["ensembles"]}
    predicted_by_key = {key(item): item for item in predicted["predictions"]}
    if len(measured_by_key) != len(measured["ensembles"]) or len(predicted_by_key) != len(predicted["predictions"]) or set(measured_by_key) != set(predicted_by_key):
        raise RuntimeError("Измеренные и модельные ключи должны точно и непусто совпадать")
    budget_path = Path(spec["uncertainty_manifest"]).expanduser().resolve()
    budget = json.loads(budget_path.read_text(encoding="utf-8"))
    if budget.get("status") != "accepted" or budget.get("experiment_id") != target:
        raise RuntimeError("Неподходящий бюджет неопределённости")
    if budget.get("measured_artifact_sha256") != sha256_file(measured_path) or budget.get("predicted_artifact_sha256") != sha256_file(predicted_path):
        raise RuntimeError("Бюджет неопределённости относится к другим версиям входов")
    entries = budget.get("entries", [])
    budget_by_key = {key(item): item for item in entries}
    if not entries or len(budget_by_key) != len(entries) or set(budget_by_key) != set(measured_by_key):
        raise RuntimeError("Бюджет должен один-к-одному покрывать все ключи")
    outputs = []
    for item_key, measured_item in measured_by_key.items():
        model_item = predicted_by_key[item_key]
        budget_item = budget_by_key[item_key]
        measured_grid = np.asarray(measured_item["time_from_r_corrected_s"], float)
        predicted_grid = np.asarray(model_item["time_from_r_corrected_s"], float)
        budget_grid = np.asarray(budget_item["time_from_r_corrected_s"], float)
        if not np.array_equal(measured_grid, predicted_grid) or not np.array_equal(measured_grid, budget_grid):
            raise RuntimeError("Временные сетки не совпали")
        result = residual_candidate(
            measured_item["mean_fractional_delta_z"], model_item["predicted_known_tissue_fractional_delta_z"],
            budget_item["measured_variance_fractional"], budget_item["predicted_variance_fractional"],
            budget_item["cross_covariance_fractional"],
        )
        outputs.append({
            **{field: measured_item.get(field) for field in KEY_FIELDS},
            "time_from_r_corrected_s": measured_grid.tolist(),
            "candidate_residual_fractional": result["residual"].tolist(),
            "variance_fractional": result["variance"].tolist(),
            "unmodeled_dynamic_regions": model_item["unmodeled_dynamic_regions"],
        })
    if not outputs:
        raise RuntimeError("Пустой выход запрещён")
    artifact = {
        "schema_version": 2, "analysis": "40.21_ttrkg_residual_candidate",
        "status": "residual_candidate_not_source_identified",
        "measured_artifact_sha256": sha256_file(measured_path), "predicted_artifact_sha256": sha256_file(predicted_path),
        "uncertainty_budget_sha256": sha256_file(budget_path),
        "forbidden_interpretations": ["validated_heart_signal", "stroke_volume", "ejection_fraction"],
        "results": outputs,
    }
    out_path = analysis_dir / "40.21_residual_candidate.json"
    out_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print("40.21 real_data_status: residual_candidate_written", out_path)


## Критерий перехода

Остаток можно передавать дальше только с полным бюджетом неопределённости и
описанием немоделированных регионов. Вычитание и ЭКГ-синхронность не доказывают
его анатомический источник.
